In [5]:
%pip install timm peft transformers

^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
print("GPU count:", torch.cuda.device_count())
print("Using:", torch.cuda.get_device_name(0))


GPU count: 1
Using: NVIDIA RTX A6000


In [4]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import math
import random
from dataclasses import dataclass
import numpy as np
from pathlib import Path
import shutil

from PIL import Image
from skimage import metrics, color

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode

import timm
from peft import LoraConfig, get_peft_model, TaskType
from transformers import Trainer, TrainingArguments

In [2]:


import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import math
import random
from dataclasses import dataclass
import numpy as np
from pathlib import Path
import shutil

from PIL import Image
from skimage import metrics, color

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode

import timm
from peft import LoraConfig, get_peft_model, TaskType
from transformers import Trainer, TrainingArguments

#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
print("VISIBLE GPUS:", os.environ["CUDA_VISIBLE_DEVICES"])
# ------------------------------------------------------
# Dataset
# ------------------------------------------------------
class MultiTaskDataset(Dataset):
    def __init__(self, root, hr_size=224, sr_scale=4):
        from torchvision.datasets import ImageFolder
        self.ds = ImageFolder(root)
        self.hr_size = hr_size
        self.sr_scale = sr_scale

        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        img, _ = self.ds[i]
        img = img.convert("RGB")
        img = transforms.Resize((self.hr_size, self.hr_size))(img)

        # --- Super Resolution (SR) ---
        lr = img.resize((self.hr_size // self.sr_scale,) * 2, Image.BICUBIC)
        lr_up = lr.resize((self.hr_size, self.hr_size), Image.BICUBIC)

        x_sr = self.normalize(self.to_tensor(lr_up))
        y_sr = self.to_tensor(img)

        # --- Colorization ---
        img_np = np.asarray(img).astype(np.float32) / 255.
        lab = color.rgb2lab(img_np)

        L = lab[..., 0:1] / 100.
        ab = lab[..., 1:3] / 128.

        L3 = np.repeat(L, 3, axis=2)

        x_col = self.normalize(torch.from_numpy(L3).permute(2, 0, 1).float())
        y_ab = torch.from_numpy(ab).permute(2, 0, 1).float()

        return {
            "sr_input": x_sr,
            "sr_target": y_sr,
            "col_input": x_col,
            "col_target": y_ab,
        }


# ------------------------------------------------------
# Decoders
# ------------------------------------------------------
class SRDecoder(nn.Module):
    def __init__(self, in_ch, scale):
        super().__init__()
        feat = 256

        self.conv1 = nn.Conv2d(in_ch, feat, 3, 1, 1)

        up = []
        for _ in range(int(math.log2(scale))):
            up += [
                nn.Conv2d(feat, feat * 4, 3, 1, 1),
                nn.PixelShuffle(2),
                nn.ReLU(True)
            ]

        self.up = nn.Sequential(*up)
        self.out = nn.Conv2d(feat, 3, 3, 1, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.up(x)
        return torch.sigmoid(self.out(x))


class ColorDecoder(nn.Module):
    def __init__(self, in_ch, up):
        super().__init__()
        feat = 128

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, feat, 3, 1, 1),
            nn.ReLU(),

            nn.Upsample(scale_factor=up, mode="bilinear", align_corners=False),

            nn.Conv2d(feat, feat, 3, 1, 1),
            nn.ReLU(),

            nn.Conv2d(feat, 2, 3, 1, 1)
        )

    def forward(self, x):
        return self.net(x)


# ------------------------------------------------------
# Shared encoder using timm + LoRA
# ------------------------------------------------------
def build_encoder(r=8, alpha=16, dropout=0.1):
    vit = timm.create_model(
        "vit_base_patch16_224.dino",
        pretrained=True,
        num_classes=0,
        global_pool=""
    )

    lora_cfg = LoraConfig(
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=["qkv", "proj", "fc1", "fc2"],
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION
    )

    vit = get_peft_model(vit, lora_cfg)
    vit.print_trainable_parameters()
    return vit


# ------------------------------------------------------
# Combined model for Trainer
# ------------------------------------------------------
class MultiTaskModel(nn.Module):
    def __init__(self, sr_scale=4):
        super().__init__()

        self.sr_scale = 4
        self.encoder = build_encoder()
        self.embed_dim = self.encoder.embed_dim

        patch = self.encoder.patch_embed.patch_size[0]

        self.sr = SRDecoder(self.embed_dim, self.sr_scale)
        self.col = ColorDecoder(self.embed_dim, up=patch)

    def forward(self, sr_input, col_input, sr_target=None, col_target=None):
        fs = self.encoder.forward_features(sr_input)
        if fs.ndim == 3:
            B, S, C = fs.shape
            h = int(math.sqrt(S))
            fs = fs[:, 1:, :].permute(0, 2, 1).reshape(B, C, h, h)

        fc = self.encoder.forward_features(col_input)
        if fc.ndim == 3:
            B, S, C = fc.shape
            h = int(math.sqrt(S))
            fc = fc[:, 1:, :].permute(0, 2, 1).reshape(B, C, h, h)

        sr_out = self.sr(fs)
        col_out = self.col(fc)

        loss = None
        if sr_target is not None and col_target is not None:
            sr_out = F.interpolate(sr_out, size=sr_target.shape[-2:], mode="bilinear", align_corners=False)
            col_out = F.interpolate(col_out, size=col_target.shape[-2:], mode="bilinear", align_corners=False)

            loss = (
                nn.functional.mse_loss(sr_out, sr_target) +
                nn.functional.mse_loss(col_out, col_target)
            )

        return {"loss": loss, "sr_out": sr_out, "col_out": col_out}


# ------------------------------------------------------
# Evaluation metrics
# ------------------------------------------------------
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    sr_pred, col_pred = preds
    sr_gt, col_gt = labels

    sr_pred = np.clip(sr_pred, 0, 1)
    sr_gt = np.clip(sr_gt, 0, 1)

    psnr = metrics.peak_signal_noise_ratio(sr_gt, sr_pred)
    ssim = metrics.structural_similarity(sr_gt, sr_pred, channel_axis=0)
    mse_col = np.mean((col_pred - col_gt) ** 2)

    return {"psnr": psnr, "ssim": ssim, "mse_color": mse_col}


# ------------------------------------------------------
# Training entry point
# ------------------------------------------------------
def run_training(root):

    # import random
    # random.seed(42)
    root = Path(root)

    # images_dir = root / "101_ObjectCategories"
    # split_root = root / "caltech_split"
    # for split in ["train", "val", "test"]:
    #     (split_root / split).mkdir(parents=True, exist_ok=True)

    # for class_dir in images_dir.iterdir():
    #     if not class_dir.is_dir():
    #         continue
    #     files = list(class_dir.glob("*.jpg"))
    #     random.shuffle(files)
    #     n = len(files)
    #     n_train = int(0.8 * n)
    #     n_val = int(0.1 * n)
    #     splits = {
    #         "train": files[:n_train],
    #         "val": files[n_train:n_train+n_val],
    #         "test": files[n_train+n_val:],
    #     }
    #     for split, fpaths in splits.items():
    #         out_class = split_root / split / class_dir.name
    #         out_class.mkdir(parents=True, exist_ok=True)
    #         for f in fpaths:
    #             shutil.copy(f, out_class / f.name)

    # print("Prepared dataset at", split_root)

    train_dir = root / "caltech_split" / "train"
    val_dir = root / "caltech_split" / "val"

    train_ds = MultiTaskDataset(train_dir)
    val_ds = MultiTaskDataset(val_dir)

    model = MultiTaskModel()

    args = TrainingArguments(
        output_dir="outputs",
        # device = "cuda:0",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=1e-4,
        eval_strategy="epoch",
        save_strategy="epoch",
        num_train_epochs=20,
        logging_steps=50,
        report_to="none",
    )

    class WrappedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            out = model(**inputs)
            return (out["loss"], out) if return_outputs else out["loss"]

    trainer = WrappedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=lambda x: compute_metrics(x)
    )

    trainer.train()


if __name__ == "__main__":
    run_training(root="data/caltech101/caltech-101")


VISIBLE GPUS: 0
trainable params: 1,191,936 || all params: 86,990,592 || trainable%: 1.3702


Epoch,Training Loss,Validation Loss
1,0.018600,No log
2,0.015600,No log
3,0.013000,No log
4,0.011200,No log
5,0.010800,No log
6,0.010400,No log
7,0.009500,No log
8,0.008700,No log
9,0.008100,No log
10,0.008100,No log


KeyboardInterrupt: 

In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, q, kv):
        B, C, H, W = q.shape

        q_flat  = q.reshape(B, C, H*W).permute(0, 2, 1)
        kv_flat = kv.reshape(B, C, H*W).permute(0, 2, 1)

        out, _ = self.attn(q_flat, kv_flat, kv_flat)
        out = self.norm(out + q_flat)

        return out.permute(0, 2, 1).reshape(B, C, H, W)

class SRDecoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        feat = 256

        self.in_conv = nn.Conv2d(dim, feat, 3, padding=1)

        # 14 → 28
        self.up1 = nn.Sequential(
            nn.Conv2d(feat, feat * 4, 3, padding=1),
            nn.PixelShuffle(2),  # 14→28
            nn.ReLU(True)
        )

        # 28 → 56
        self.up2 = nn.Sequential(
            nn.Conv2d(feat, feat * 4, 3, padding=1),
            nn.PixelShuffle(2),  # 28→56
            nn.ReLU(True)
        )

        # 56 → 112
        self.up3 = nn.Sequential(
            nn.Conv2d(feat, feat * 4, 3, padding=1),
            nn.PixelShuffle(2),  # 56→112
            nn.ReLU(True)
        )

        # 112 → 224
        self.up4 = nn.Sequential(
            nn.Conv2d(feat, feat * 4, 3, padding=1),
            nn.PixelShuffle(2),  # 112→224
            nn.ReLU(True)
        )

        self.out = nn.Conv2d(feat, 3, 3, padding=1)

    def forward(self, x):
        feats = {}  # store features for cross-attn
        x = F.relu(self.in_conv(x))

        x = self.up1(x); feats["28"] = x
        x = self.up2(x); feats["56"] = x
        x = self.up3(x); feats["112"] = x
        x = self.up4(x); feats["224"] = x

        out = torch.sigmoid(self.out(x))

        return out, feats


class ColorDecoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        feat = 256

        self.in_conv = nn.Conv2d(dim, feat, 3, padding=1)

        # 14 → 28
        self.up1 = nn.Sequential(
            nn.Conv2d(feat, feat * 4, 3, padding=1),
            nn.PixelShuffle(2),
            nn.ReLU(True),
        )

        # 28 → 56
        self.up2 = nn.Sequential(
            nn.Conv2d(feat, feat * 4, 3, padding=1),
            nn.PixelShuffle(2),
            nn.ReLU(True),
        )

        # 56 → 112
        self.up3 = nn.Sequential(
            nn.Conv2d(feat, feat * 4, 3, padding=1),
            nn.PixelShuffle(2),
            nn.ReLU(True),
        )

        # 112 → 224
        self.up4 = nn.Sequential(
            nn.Conv2d(feat, feat * 4, 3, padding=1),
            nn.PixelShuffle(2),
            nn.ReLU(True),
        )

        # final prediction: 2 channels (a,b)
        self.out = nn.Conv2d(feat, 2, 3, padding=1)

    def forward(self, x):
        feats = {}

        x = F.relu(self.in_conv(x))

        x = self.up1(x); feats["28"]  = x
        x = self.up2(x); feats["56"]  = x
        x = self.up3(x); feats["112"] = x
        x = self.up4(x); feats["224"] = x

        out = self.out(x)
        return out, feats

class MultiTaskModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = build_encoder()
        dim = self.encoder.embed_dim

        self.sr = SRDecoder(dim)
        self.col = ColorDecoder(dim)

        # 4 levels of decoder attention
        self.ca28  = CrossAttention(256)
        self.ca56  = CrossAttention(256)
        self.ca112 = CrossAttention(256)
        self.ca224 = CrossAttention(256)

    def _vit_fmap(self, x):
        f = self.encoder.forward_features(x)
        B, S, C = f.shape
        H = int((S-1)**0.5)
        return f[:,1:,:].permute(0,2,1).reshape(B,C,H,H)

    def forward(self, sr_input, col_input, sr_target=None, col_target=None):
        # Encoder features
        fs = self._vit_fmap(sr_input)
        fc = self._vit_fmap(col_input)

        # Decoder forward passes (collect multi-resolution feats)
        sr_out,  sr_feats  = self.sr(fs)
        col_out, col_feats = self.col(fc)

        # ---- Cross-attention at each resolution ----
        sr_feats["28"]  = self.ca28 (sr_feats["28"],  col_feats["28"])
        col_feats["28"] = self.ca28 (col_feats["28"], sr_feats["28"])

        sr_feats["56"]  = self.ca56 (sr_feats["56"],  col_feats["56"])
        col_feats["56"] = self.ca56 (col_feats["56"], sr_feats["56"])

        sr_feats["112"]  = self.ca112(sr_feats["112"], col_feats["112"])
        col_feats["112"] = self.ca112(col_feats["112"], sr_feats["112"])

        sr_feats["224"]  = self.ca224(sr_feats["224"], col_feats["224"])
        col_feats["224"] = self.ca224(col_feats["224"], sr_feats["224"])

        # Loss
        loss = None
        if sr_target is not None:
            loss = F.mse_loss(sr_out, sr_target)

        if col_target is not None:
            loss = loss + F.mse_loss(col_out, col_target)

        return {
            "loss": loss,
            "sr_out": sr_out,
            "col_out": col_out
        }


In [5]:
class MultiTaskDataset(Dataset):
    def __init__(self, root, hr_size=224, sr_scale=4):
        from torchvision.datasets import ImageFolder
        self.ds = ImageFolder(root)
        self.hr_size = hr_size
        self.sr_scale = sr_scale

        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        img, _ = self.ds[i]
        img = img.convert("RGB")
        img = transforms.Resize((self.hr_size, self.hr_size))(img)

        # --- Super Resolution (SR) ---
        lr = img.resize((self.hr_size // self.sr_scale,) * 2, Image.BICUBIC)
        lr_up = lr.resize((self.hr_size, self.hr_size), Image.BICUBIC)

        x_sr = self.normalize(self.to_tensor(lr_up))
        y_sr = self.to_tensor(img)

        # --- Colorization ---
        img_np = np.asarray(img).astype(np.float32) / 255.
        lab = color.rgb2lab(img_np)

        L = lab[..., 0:1] / 100.
        ab = lab[..., 1:3] / 128.

        L3 = np.repeat(L, 3, axis=2)

        x_col = self.normalize(torch.from_numpy(L3).permute(2, 0, 1).float())
        y_ab = torch.from_numpy(ab).permute(2, 0, 1).float()

        return {
            "sr_input": x_sr,
            "sr_target": y_sr,
            "col_input": x_col,
            "col_target": y_ab,
        }


# ------------------------------------------------------
# Decoders
# ------------------------------------------------------
class SRDecoder(nn.Module):
    def __init__(self, in_ch, scale):
        super().__init__()
        feat = 256

        self.conv1 = nn.Conv2d(in_ch, feat, 3, 1, 1)

        up = []
        for _ in range(int(math.log2(scale))):
            up += [
                nn.Conv2d(feat, feat * 4, 3, 1, 1),
                nn.PixelShuffle(2),
                nn.ReLU(True)
            ]

        self.up = nn.Sequential(*up)
        self.out = nn.Conv2d(feat, 3, 3, 1, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.up(x)
        return torch.sigmoid(self.out(x))


class ColorDecoder(nn.Module):
    def __init__(self, in_ch, up):
        super().__init__()
        feat = 128

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, feat, 3, 1, 1),
            nn.ReLU(),

            nn.Upsample(scale_factor=up, mode="bilinear", align_corners=False),

            nn.Conv2d(feat, feat, 3, 1, 1),
            nn.ReLU(),

            nn.Conv2d(feat, 2, 3, 1, 1)
        )

    def forward(self, x):
        return self.net(x)


# ------------------------------------------------------
# Shared encoder using timm + LoRA
# ------------------------------------------------------
def build_encoder(r=8, alpha=16, dropout=0.1):
    vit = timm.create_model(
        "vit_base_patch16_224.dino",
        pretrained=True,
        num_classes=0,
        global_pool=""
    )

    lora_cfg = LoraConfig(
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=["qkv", "proj", "fc1", "fc2"],
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION
    )

    vit = get_peft_model(vit, lora_cfg)
    vit.print_trainable_parameters()
    return vit


# ------------------------------------------------------
# Combined model for Trainer
# ------------------------------------------------------
class MultiTaskModel(nn.Module):
    def __init__(self, sr_scale=4):
        super().__init__()

        self.sr_scale = 4
        self.encoder = build_encoder()
        self.embed_dim = self.encoder.embed_dim

        patch = self.encoder.patch_embed.patch_size[0]

        self.sr = SRDecoder(self.embed_dim, self.sr_scale)
        self.col = ColorDecoder(self.embed_dim, up=patch)

    def forward(self, sr_input, col_input, sr_target=None, col_target=None):
        fs = self.encoder.forward_features(sr_input)
        if fs.ndim == 3:
            B, S, C = fs.shape
            h = int(math.sqrt(S))
            fs = fs[:, 1:, :].permute(0, 2, 1).reshape(B, C, h, h)

        fc = self.encoder.forward_features(col_input)
        if fc.ndim == 3:
            B, S, C = fc.shape
            h = int(math.sqrt(S))
            fc = fc[:, 1:, :].permute(0, 2, 1).reshape(B, C, h, h)

        sr_out = self.sr(fs)
        col_out = self.col(fc)

        loss = None
        if sr_target is not None and col_target is not None:
            sr_out = F.interpolate(sr_out, size=sr_target.shape[-2:], mode="bilinear", align_corners=False)
            col_out = F.interpolate(col_out, size=col_target.shape[-2:], mode="bilinear", align_corners=False)

            loss = (
                nn.functional.mse_loss(sr_out, sr_target) +
                nn.functional.mse_loss(col_out, col_target)
            )

        return {"loss": loss, "sr_out": sr_out, "col_out": col_out}

In [18]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from skimage import metrics, color
from PIL import Image
import matplotlib.pyplot as plt
import os
from safetensors.torch import load_file

# ------------------------------------------------------
# 1. LOAD TRAINED MODEL
# ------------------------------------------------------
def load_trained_model(checkpoint_path="outputs/checkpoint-best"):
    model = MultiTaskModel()
    state_dict = load_file(f"{checkpoint_path}/model.safetensors")
    new_state = {}
    for k, v in state_dict.items():
        new_key = k.replace("model.", "") if k.startswith("model.") else k
        new_state[new_key] = v

    model.load_state_dict(new_state)
    model.eval()
#     model.load_state_dict(torch.load(f"{checkpoint_path}/model.safetensors"))
#     model.eval()
    return model


# ------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------
def tensor_to_img(t):
    """Convert 3xHxW tensor → numpy RGB (0-1)"""
    t = t.detach().cpu().numpy()
    t = np.clip(t, 0, 1)
    return np.transpose(t, (1, 2, 0))


def lab_Lab_to_rgb(L, ab):
    """Convert lightness + ab channels → RGB"""
    #print(ab.shape)
    L = L.unsqueeze(0)
    L = L.detach().cpu().numpy()
    ab = ab.detach().cpu().numpy()
    
    L = np.transpose(L, (1, 2,0)) * 100
    ab = np.transpose(ab, (1, 2,0)) * 128

    lab = np.concatenate([L, ab], axis=-1)
    rgb = color.lab2rgb(lab)
    return np.clip(rgb, 0, 1)


# ------------------------------------------------------
# 3. RUN INFERENCE ON FEW IMAGES
# ------------------------------------------------------
def run_inference(model, dataset, num_samples=4, device="cuda:0"):

    os.makedirs("inference_outputs", exist_ok=True)

    model = model.to(device)
    model.eval()

    # pick N random samples
    indices = np.random.choice(len(dataset), num_samples, replace=False)

    for i, idx in enumerate(indices):
        batch = dataset[idx]
        #print(batch["col_input"].shape)
        sr_in = batch["sr_input"].unsqueeze(0).to(device)
        col_in = batch["col_input"].unsqueeze(0).to(device)

        sr_gt = batch["sr_target"]
        col_gt = batch["col_target"]

        with torch.no_grad():
            out = model(sr_input=sr_in, col_input=col_in,sr_target=sr_gt.unsqueeze(0).to(device),col_target=col_gt.unsqueeze(0).to(device))
            sr_pred = out["sr_out"][0]
            col_pred = out["col_out"][0]

        # Convert tensors → images
        sr_input_vis = tensor_to_img(sr_in[0])
        sr_gt_img = tensor_to_img(sr_gt)
        sr_pred_img = tensor_to_img(sr_pred)

        col_input_vis = lab_Lab_to_rgb(col_in[0],
        col_gt_img = lab_Lab_to_rgb(batch["col_input"][0], col_gt)
        col_pred_img = lab_Lab_to_rgb(batch["col_input"][0], col_pred)

        # Compute metrics
        # print(sr_gt_img.shape)
        # print(sr_pred_img.shape)
        psnr = metrics.peak_signal_noise_ratio(sr_gt_img, sr_pred_img)
        ssim = metrics.structural_similarity(sr_gt_img, sr_pred_img, channel_axis=2,data_range=1.0)
        mse_col = np.mean((col_pred_img - col_gt_img) ** 2)

        print(f"\n=== Sample {i} (index {idx}) ===")
        print(f"PSNR:      {psnr:.3f}")
        print(f"SSIM:      {ssim:.3f}")
        print(f"MSE Color: {mse_col:.6f}")

        # -------------------------------
        # Plot the results
        # -------------------------------
        fig, ax = plt.subplots(2, 2, figsize=(10, 8))
        fig.suptitle(f"Sample {i} | PSNR={psnr:.2f}, SSIM={ssim:.2f}")

        ax[0, 0].imshow(sr_gt_img)
        ax[0, 0].set_title("Ground Truth HR")
        ax[0, 0].axis("off")

        ax[0, 1].imshow(sr_pred_img)
        ax[0, 1].set_title("SR Output")
        ax[0, 1].axis("off")

        ax[1, 0].imshow(col_gt_img)
        ax[1, 0].set_title("Color GT (LAB→RGB)")
        ax[1, 0].axis("off")

        ax[1, 1].imshow(col_pred_img)
        ax[1, 1].set_title("Colorization Output")
        ax[1, 1].axis("off")

        plt.tight_layout()
        plt.savefig(f"inference_outputs/sample_{i}.png")
        plt.close()


# ------------------------------------------------------
# 4. MAIN
# ------------------------------------------------------
if __name__ == "__main__":
    # change this if your test set path is different
    test_dir = "data/caltech101/caltech-101/caltech_split/test"
    test_ds = MultiTaskDataset(test_dir)

    model = load_trained_model("outputs/checkpoint-5005")
    run_inference(model, test_ds, num_samples=4)


trainable params: 1,191,936 || all params: 86,990,592 || trainable%: 1.3702
(224, 224, 3)
(224, 224, 3)


/tmp/ipykernel_2549567/540960476.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 31690 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)
/tmp/ipykernel_2549567/540960476.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 31811 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)



=== Sample 0 (index 237) ===
PSNR:      18.947
SSIM:      0.686
MSE Color: 0.000131


/tmp/ipykernel_2549567/540960476.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 10171 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)
/tmp/ipykernel_2549567/540960476.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 10430 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)


(224, 224, 3)
(224, 224, 3)

=== Sample 1 (index 360) ===
PSNR:      21.846
SSIM:      0.762
MSE Color: 0.000641


/tmp/ipykernel_2549567/540960476.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 15847 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)
/tmp/ipykernel_2549567/540960476.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 16171 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)


(224, 224, 3)
(224, 224, 3)

=== Sample 2 (index 66) ===
PSNR:      24.360
SSIM:      0.760
MSE Color: 0.001067


/tmp/ipykernel_2549567/540960476.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 4363 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)
/tmp/ipykernel_2549567/540960476.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 4330 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)


(224, 224, 3)
(224, 224, 3)

=== Sample 3 (index 301) ===
PSNR:      29.164
SSIM:      0.893
MSE Color: 0.001701


In [16]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from skimage import metrics, color
from PIL import Image
import matplotlib.pyplot as plt
import os
from safetensors.torch import load_file

# ------------------------------------------------------
# 1. LOAD TRAINED MODEL
# ------------------------------------------------------
def load_trained_model(checkpoint_path="outputs/checkpoint-best"):
    model = MultiTaskModel()
    state_dict = load_file(f"{checkpoint_path}/model.safetensors")
    new_state = {}
    for k, v in state_dict.items():
        new_key = k.replace("model.", "") if k.startswith("model.") else k
        new_state[new_key] = v

    model.load_state_dict(new_state)
    model.eval()
#     model.load_state_dict(torch.load(f"{checkpoint_path}/model.safetensors"))
#     model.eval()
    return model


# ------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------
def tensor_to_img(t):
    """Convert 3xHxW tensor → numpy RGB (0-1)"""
    t = t.detach().cpu().numpy()
    t = np.clip(t, 0, 1)
    return np.transpose(t, (1, 2, 0))


def lab_Lab_to_rgb(L, ab):
    """Convert lightness + ab channels → RGB"""
    #print(ab.shape)
    L = L.unsqueeze(0)
    L = L.detach().cpu().numpy()
    ab = ab.detach().cpu().numpy()
    
    L = np.transpose(L, (1, 2,0)) * 100
    ab = np.transpose(ab, (1, 2,0)) * 128

    lab = np.concatenate([L, ab], axis=-1)
    rgb = color.lab2rgb(lab)
    return np.clip(rgb, 0, 1)

def rgb_to_bw(x):
    # x: (3, H, W), range [0,1]
    r, g, b = x[0], x[1], x[2]
    bw = 0.2989 * r + 0.5870 * g + 0.1140 * b   # standard luminance
    return bw  # shape: (H, W)

# ------------------------------------------------------
# 3. RUN INFERENCE ON FEW IMAGES
# ------------------------------------------------------
def run_inference(model, dataset, num_samples=4, device="cuda:0"):

    os.makedirs("inference_outputs-1", exist_ok=True)

    model = model.to(device)
    model.eval()

    # pick N random samples
    indices = np.random.choice(len(dataset), num_samples, replace=False)

    for i, idx in enumerate(indices):
        batch = dataset[idx]
        #print(batch["col_input"].shape)
        sr_in = batch["sr_input"].unsqueeze(0).to(device)
        col_in = batch["col_input"].unsqueeze(0).to(device)

        sr_gt = batch["sr_target"]
        col_gt = batch["col_target"]

        with torch.no_grad():
            out = model(sr_input=sr_in, col_input=col_in,sr_target=sr_gt.unsqueeze(0).to(device),col_target=col_gt.unsqueeze(0).to(device))
            sr_pred = out["sr_out"][0]
            col_pred = out["col_out"][0]

        # Convert tensors → images
        sr_input_vis = tensor_to_img(sr_in[0])
        sr_gt_img = tensor_to_img(sr_gt)
        sr_pred_img = tensor_to_img(sr_pred)

        col_input_vis = tensor_to_img(rgb_to_bw(col_in[0]).unsqueeze(0))#.permute(2,0,1))
        print(col_input_vis.shape)
        col_gt_img = lab_Lab_to_rgb(batch["col_input"][0], col_gt)
        col_pred_img = lab_Lab_to_rgb(batch["col_input"][0], col_pred)

        # Compute metrics
        # print(sr_gt_img.shape)
        # print(sr_pred_img.shape)
        psnr = metrics.peak_signal_noise_ratio(sr_gt_img, sr_pred_img)
        ssim = metrics.structural_similarity(sr_gt_img, sr_pred_img, channel_axis=2,data_range=1.0)
        mse_col = np.mean((col_pred_img - col_gt_img) ** 2)

        print(f"\n=== Sample {i} (index {idx}) ===")
        print(f"PSNR:      {psnr:.3f}")
        print(f"SSIM:      {ssim:.3f}")
        print(f"MSE Color: {mse_col:.6f}")

        # -------------------------------
        # Plot the results
        # -------------------------------
        fig, ax = plt.subplots(2, 3, figsize=(10, 8))
        fig.suptitle(f"Sample {i} | PSNR={psnr:.2f}, SSIM={ssim:.2f}")

        ax[0, 0].imshow(sr_gt_img)
        ax[0, 0].set_title("Ground Truth HR")
        ax[0, 0].axis("off")

        ax[0, 1].imshow(sr_pred_img)
        ax[0, 1].set_title("SR Output")
        ax[0, 1].axis("off")

        ax[0, 2].imshow(sr_input_vis)
        ax[0, 2].set_title("SR input")
        ax[0, 2].axis("off")

        ax[1, 0].imshow(col_gt_img)
        ax[1, 0].set_title("Color GT (LAB→RGB)")
        ax[1, 0].axis("off")

        ax[1, 1].imshow(col_pred_img)
        ax[1, 1].set_title("Colorization Output")
        ax[1, 1].axis("off")

        ax[1, 2].imshow(col_input_vis,cmap='grey')
        ax[1, 2].set_title("Colorization input")
        ax[1, 2].axis("off")

        plt.tight_layout()
        plt.savefig(f"inference_outputs-1/sample_{i}.png")
        plt.close()


# ------------------------------------------------------
# 4. MAIN
# ------------------------------------------------------
if __name__ == "__main__":
    # change this if your test set path is different
    test_dir = "data/caltech101/caltech-101/caltech_split/test"
    test_ds = MultiTaskDataset(test_dir)

    model = load_trained_model("outputs/checkpoint-5005")
    run_inference(model, test_ds, num_samples=4)


trainable params: 1,191,936 || all params: 86,990,592 || trainable%: 1.3702
(224, 224, 1)

=== Sample 0 (index 928) ===
PSNR:      26.862
SSIM:      0.703
MSE Color: 0.002109


/tmp/ipykernel_2801913/2735799383.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 11492 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)
/tmp/ipykernel_2801913/2735799383.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 11239 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)


(224, 224, 1)

=== Sample 1 (index 977) ===
PSNR:      19.548
SSIM:      0.632
MSE Color: 0.000431


/tmp/ipykernel_2801913/2735799383.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 31512 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)
/tmp/ipykernel_2801913/2735799383.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 31660 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)


(224, 224, 1)

=== Sample 2 (index 668) ===
PSNR:      24.731
SSIM:      0.755
MSE Color: 0.000755


/tmp/ipykernel_2801913/2735799383.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 16102 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)
/tmp/ipykernel_2801913/2735799383.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 15871 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)


(224, 224, 1)

=== Sample 3 (index 482) ===
PSNR:      22.770
SSIM:      0.612
MSE Color: 0.000943


/tmp/ipykernel_2801913/2735799383.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 18835 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)
/tmp/ipykernel_2801913/2735799383.py:49: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 19350 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)


In [ ]:
#set-5

In [33]:
import torch
import numpy as np
import os
from PIL import Image
from skimage import metrics, color
import matplotlib.pyplot as plt
from safetensors.torch import load_file


# ------------------------------------------------------
# Load trained model
# ------------------------------------------------------
def load_trained_model(checkpoint_path):
    model = MultiTaskModel()
    state_dict = load_file(f"{checkpoint_path}/model.safetensors")

    clean_state = {}
    for k, v in state_dict.items():
        new_k = k.replace("model.", "") if k.startswith("model.") else k
        clean_state[new_k] = v

    model.load_state_dict(clean_state)
    model.eval()
    return model


# ------------------------------------------------------
# Helpers
# ------------------------------------------------------
def tensor_to_img(t):
    """Convert tensor 3×H×W → float image H×W×3"""
    t = t.detach().cpu().numpy()
    t = np.clip(t, 0, 1)
    return np.transpose(t, (1, 2, 0))


def lab_to_rgb(L, ab):
    """L tensor: 1×H×W, ab tensor: 2×H×W"""
    L = L.detach().cpu().numpy().transpose(1, 2, 0) * 100
    ab = ab.detach().cpu().numpy().transpose(1, 2, 0) * 128
    lab = np.concatenate([L, ab], axis=-1)
    rgb = color.lab2rgb(lab)
    return np.clip(rgb, 0, 1)


# ------------------------------------------------------
# MAIN inference for SR + Colorization
# ------------------------------------------------------
def run_set5_inference(model, set5_dir="data/set-5", device="cuda:0"):

    model = model.to(device).eval()
    os.makedirs("inference_set5_outputs", exist_ok=True)

    images = sorted([f for f in os.listdir(set5_dir) if f.lower().endswith(("png","jpg","jpeg"))])

    for img_name in images:
        print(f"\n=== Processing {img_name} ===")

        # ----------------------------------------------------------
        # Load HR image (convert to 224x224)
        # ----------------------------------------------------------
        img_pil = Image.open(os.path.join(set5_dir, img_name)).convert("RGB")
        hr_pil = img_pil.resize((224, 224), Image.BICUBIC)
        hr = np.array(hr_pil) / 255.0                     # (224,224,3)

        # HR tensor
        hr_t = torch.tensor(hr.transpose(2, 0, 1), dtype=torch.float32)

        # ----------------------------------------------------------
        # Create LR (downscale by 4 then resize to 224)
        # ----------------------------------------------------------
        lr_pil_small = hr_pil.resize((224//4, 224//4), Image.BICUBIC)     # → small LR
        lr_pil = lr_pil_small.resize((224, 224), Image.BICUBIC)           # upscale back
        lr = np.array(lr_pil) / 255.0
        lr_t = torch.tensor(lr.transpose(2, 0, 1), dtype=torch.float32)

        # ----------------------------------------------------------
        # Colorization input (L-channel only)
        # ----------------------------------------------------------
        lab = color.rgb2lab(hr)        # convert HR image to LAB

        L = lab[..., 0:1] / 100.0      # (H,W,1)
        ab = lab[..., 1:] / 128.0      # (H,W,2)

        # Expand L to 3 channels because your model expects 3×H×W
        L3 = np.repeat(L, 3, axis=2)   # (H,W,3)
        L3_t = torch.tensor(L3.transpose(2, 0, 1), dtype=torch.float32)

        # ----------------------------------------------------------
        # Run model
        # ----------------------------------------------------------
        with torch.no_grad():
            out = model(
                sr_input=lr_t.unsqueeze(0).to(device),
                col_input=L3_t.unsqueeze(0).to(device), 
                sr_target=hr_t.unsqueeze(0).to(device),    # (1,3,224,224)
        col_target=torch.tensor(ab.transpose(2, 0, 1), dtype=torch.float32)
                    .unsqueeze(0).to(device)       # (1,2,224,224)
            )

            sr_pred = out["sr_out"][0].cpu()
            col_pred = out["col_out"][0].cpu()

        # ----------------------------------------------------------
        # Convert outputs
        # ----------------------------------------------------------
        sr_pred_img = tensor_to_img(sr_pred)
        bw_img = L3  # grayscale expanded to 3 channels
        color_pred_img = lab_to_rgb(
            torch.tensor(L.transpose(2, 0, 1), dtype=torch.float32),
            col_pred
        )

        # ----------------------------------------------------------
        # Metrics
        # ----------------------------------------------------------
        print(sr_pred_img.shape)
        print(hr.shape)
        psnr = metrics.peak_signal_noise_ratio(hr, sr_pred_img)
        ssim = metrics.structural_similarity(hr, sr_pred_img, channel_axis=2,data_range=1)
        mse_color = np.mean((color_pred_img - hr)**2)

        print(f"PSNR: {psnr:.3f},  SSIM: {ssim:.3f},  Color MSE: {mse_color:.6f}")

        # ----------------------------------------------------------
        # Plot and save
        # ----------------------------------------------------------
        fig, ax = plt.subplots(1, 5, figsize=(20, 4))
        fig.suptitle(f"{img_name}\nPSNR={psnr:.2f} | SSIM={ssim:.2f} | MSE={mse_color:.6f}")

        ax[0].imshow(hr); ax[0].set_title("Ground Truth"); ax[0].axis("off")
        ax[1].imshow(bw_img); ax[1].set_title("Grayscale Input"); ax[1].axis("off")
        ax[2].imshow(color_pred_img); ax[2].set_title("Colorized Prediction"); ax[2].axis("off")
        ax[3].imshow(lr); ax[3].set_title("LR (224x224)"); ax[3].axis("off")
        ax[4].imshow(sr_pred_img); ax[4].set_title("SR Prediction"); ax[4].axis("off")

        plt.tight_layout()
        plt.savefig(f"inference_set5_outputs/{img_name}.png")
        plt.close()


# ------------------------------------------------------
# Entry point
# ------------------------------------------------------
if __name__ == "__main__":
    model = load_trained_model("outputs/checkpoint-5005")
    run_set5_inference(model)


trainable params: 1,191,936 || all params: 86,990,592 || trainable%: 1.3702

=== Processing baby.png ===
(224, 224, 3)
(224, 224, 3)
PSNR: 12.925,  SSIM: 0.488,  Color MSE: 0.008863


/tmp/ipykernel_2801913/1854914754.py:119: UserWarning: Inputs have mismatched dtype.  Setting data_range based on image_true.
  psnr = metrics.peak_signal_noise_ratio(hr, sr_pred_img)



=== Processing bird.png ===
(224, 224, 3)
(224, 224, 3)
PSNR: 10.717,  SSIM: 0.292,  Color MSE: 0.029460

=== Processing butterfly.png ===
(224, 224, 3)
(224, 224, 3)
PSNR: 12.745,  SSIM: 0.412,  Color MSE: 0.013967


/tmp/ipykernel_2801913/1854914754.py:42: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 7 negative Z values that have been clipped to zero
  rgb = color.lab2rgb(lab)



=== Processing head.png ===
(224, 224, 3)
(224, 224, 3)
PSNR: 10.407,  SSIM: 0.323,  Color MSE: 0.006605

=== Processing woman.png ===
(224, 224, 3)
(224, 224, 3)
PSNR: 12.421,  SSIM: 0.426,  Color MSE: 0.004981
